In [3]:
import torch
print(torch.cuda.is_available())  # Returns True if GPU is available   

True


In [5]:
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")   

Tesla T4
15.637086208 GB


In [3]:
%pip install transformers datasets accelerate torch torchvision

In [6]:
from datasets import load_dataset, Image

dataset = load_dataset("Sandesh-Lav/potato-tuber-caption-dataset")

# decode images
dataset = dataset.cast_column("image", Image())

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['image', 'caption'],
        num_rows: 1661
    })
    validation: Dataset({
        features: ['image', 'caption'],
        num_rows: 92
    })
    test: Dataset({
        features: ['image', 'caption'],
        num_rows: 93
    })
})


In [7]:
from transformers import BlipProcessor, BlipForConditionalGeneration

model_name = "Salesforce/blip-image-captioning-base"

processor = BlipProcessor.from_pretrained(model_name)
model = BlipForConditionalGeneration.from_pretrained(model_name)

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

In [8]:
def preprocess(example):
    
    encoding = processor(
        images=example["image"],
        text=example["caption"],
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    encoding = {k: v.squeeze() for k, v in encoding.items()}
    encoding["labels"] = encoding["input_ids"]

    return encoding


train_ds = dataset["train"].map(preprocess, remove_columns=dataset["train"].column_names)
val_ds = dataset["validation"].map(preprocess, remove_columns=dataset["validation"].column_names)

Map:   0%|          | 0/1661 [00:00<?, ? examples/s]

Map:   0%|          | 0/92 [00:00<?, ? examples/s]

In [ ]:
%pip install -U transformers accelerate

In [9]:
import transformers
print(transformers.__version__)

5.3.0


In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    
    output_dir="./blip-potato-caption",
    
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    
    learning_rate=5e-5,
    
    num_train_epochs=5,
    
    logging_steps=50,
    
    fp16=True,
    
    remove_unused_columns=False
)

In [ ]:
%pip install -U transformers accelerate datasets
%pip uninstall -y deepspeed

In [11]:
train_ds

Dataset({
    features: ['pixel_values', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 1661
})

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

trainer.train()

Step,Training Loss
50,6.716351
100,1.084934
150,0.021952
200,0.007179
250,0.004939
300,0.005336
350,0.004828
400,0.003848
450,0.003740
500,0.003490


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2080, training_loss=0.19469765765556635, metrics={'train_runtime': 3666.3037, 'train_samples_per_second': 2.265, 'train_steps_per_second': 0.567, 'total_flos': 4.929056278287483e+18, 'train_loss': 0.19469765765556635, 'epoch': 5.0})

In [24]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)
model.eval()

sample = dataset["test"][0]
image = sample["image"]

with torch.no_grad():

    inputs = processor(images=image, return_tensors="pt")

    # move tensors to GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out = model.generate(
        **inputs,
        max_new_tokens=40   # removes warning about max_length
    )

caption = processor.decode(out[0], skip_special_tokens=True)

print("Generated:", caption)
print("Actual:", sample["caption"])

image.show()

Generated: a cut - section of a potato tuber showing moderate dry rot within the internal tissue.
Actual: A cut-section of a potato tuber showing moderate dry rot within the internal tissue.


In [ ]:
from huggingface_hub import login


login()

In [17]:
trainer.save_model("blip-potato-caption-model")
processor.save_pretrained("blip-potato-caption-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['blip-potato-caption-model/processor_config.json']

In [18]:
trainer.push_to_hub("potato-caption-blip")
processor.push_to_hub("potato-caption-blip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...caption/model.safetensors:   2%|1         | 19.7MB /  990MB            

  ...caption/training_args.bin:  13%|#3        |   681B / 5.20kB            

CommitInfo(commit_url='https://huggingface.co/Sandesh-Lav/potato-caption-blip/commit/f389cb9ab77a2fef6cbfec32c8819367c468f2b8', commit_message='Upload processor', commit_description='', oid='f389cb9ab77a2fef6cbfec32c8819367c468f2b8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sandesh-Lav/potato-caption-blip', endpoint='https://huggingface.co', repo_type='model', repo_id='Sandesh-Lav/potato-caption-blip'), pr_revision=None, pr_num=None)

In [19]:
trainer.save_model("blip-potato-caption-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
trainer.push_to_hub("potato-caption-blip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...caption/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...caption/model.safetensors:   4%|4         | 39.9MB /  990MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Sandesh-Lav/blip-potato-caption/commit/44803c94d381d0e11addc98f18aca3149ca36ffb', commit_message='potato-caption-blip', commit_description='', oid='44803c94d381d0e11addc98f18aca3149ca36ffb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sandesh-Lav/blip-potato-caption', endpoint='https://huggingface.co', repo_type='model', repo_id='Sandesh-Lav/blip-potato-caption'), pr_revision=None, pr_num=None)

In [21]:
trainer.save_model("blip-potato-caption-model")
processor.save_pretrained("blip-potato-caption-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['blip-potato-caption-model/processor_config.json']

In [22]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="blip-potato-caption-model",
    repo_id="Sandesh-Lav/potato-caption-blip",
    repo_type="model"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n-model/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...n-model/model.safetensors:   4%|4         | 39.9MB /  990MB            

CommitInfo(commit_url='https://huggingface.co/Sandesh-Lav/potato-caption-blip/commit/275111ea8847ffca6672d9c30dbedb1206e89a39', commit_message='Upload folder using huggingface_hub', commit_description='', oid='275111ea8847ffca6672d9c30dbedb1206e89a39', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sandesh-Lav/potato-caption-blip', endpoint='https://huggingface.co', repo_type='model', repo_id='Sandesh-Lav/potato-caption-blip'), pr_revision=None, pr_num=None)

In [23]:
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Sandesh-Lav/potato-caption-blip")
model = BlipForConditionalGeneration.from_pretrained("Sandesh-Lav/potato-caption-blip")

processor_config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

In [26]:
from transformers import pipeline

pipe = pipeline(
    "image-text-to-text",
    model="Sandesh-Lav/potato-caption-blip"
)

pipe("potato.jpg")

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


TypeError: BatchEncoding.to() got an unexpected keyword argument 'dtype'

In [1]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

processor = BlipProcessor.from_pretrained("Sandesh-Lav/potato-caption-blip")
model = BlipForConditionalGeneration.from_pretrained("Sandesh-Lav/potato-caption-blip")

image = Image.open("image.png").convert("RGB")

inputs = processor(images=image, return_tensors="pt")

out = model.generate(**inputs, max_new_tokens=40)

caption = processor.decode(out[0], skip_special_tokens=True)

print(caption)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


a whole potato tuber showing severe external manifestation of bacterial soft rot with extensive tissue degradation.


In [ ]:
%pip install google-api-python-client

In [30]:
%pip install google-colab

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Benchmark Metrics

In [35]:
%pip install evaluate nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00


In [36]:
import evaluate
import torch
from tqdm import tqdm

In [ ]:
%pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=b46b2409279af816673be6d3113496593697bb4c62986bd86102b7d4e4efd2c4
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [39]:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [40]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

predictions = []
references = []

for sample in tqdm(dataset["test"]):

    image = sample["image"]
    ref_caption = sample["caption"]

    inputs = processor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=40)

    pred_caption = processor.decode(out[0], skip_special_tokens=True)

    predictions.append(pred_caption)
    references.append([ref_caption])

100%|██████████| 93/93 [00:39<00:00,  2.33it/s]


In [41]:
bleu_score = bleu.compute(predictions=predictions, references=references)
rouge_score = rouge.compute(predictions=predictions, references=[r[0] for r in references])
meteor_score = meteor.compute(predictions=predictions, references=[r[0] for r in references])

In [42]:
print("BLEU:", bleu_score["bleu"])
print("ROUGE-L:", rouge_score["rougeL"])
print("METEOR:", meteor_score["meteor"])

BLEU: 0.7432517455331193
ROUGE-L: 0.9552267635137488
METEOR: 0.9215191311789924


In [43]:
import pandas as pd

df = pd.DataFrame({
    "prediction": predictions,
    "reference": [r[0] for r in references]
})

df.to_csv("caption_results.csv", index=False)

In [44]:
%pip install pycocoevalcap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 10.0 MB/s eta 0:00:0000:0100:01


In [45]:
from pycocoevalcap.cider.cider import Cider

In [46]:
gts = {}
res = {}

for i in range(len(predictions)):
    gts[i] = references[i]          # references already like ["caption"]
    res[i] = [predictions[i]]

In [47]:
cider_scorer = Cider()

score, scores = cider_scorer.compute_score(gts, res)

print("CIDEr:", score)

CIDEr: 5.429658607771995


out = model.generate(
    **inputs,
    max_new_tokens=40,
    num_beams=5,
    length_penalty=1.0,
    early_stopping=True
)